In [1]:
import math

In [2]:
from dataclasses import dataclass
import numpy as np

@dataclass
class CombinedResult:
    R_comb: float                 # combined central value
    weights: np.ndarray           # BLUE weights (sum to 1)
    sigma_stat: float             # absolute statistical uncertainty
    sigma_sys_unc: float          # absolute uncorrelated systematic
    sigma_sys_com: float          # absolute fully-correlated (multiplicative) systematic
    sigma_sys_tot: float          # sqrt(sys_unc^2 + sys_com^2)
    sigma_tot: float              # sqrt(stat^2 + sys_tot^2)

def _build_common_cov(R: np.ndarray, c_common):
    """
    Build the covariance matrix for fully-correlated multiplicative systematics.
    - If c_common is a scalar: uses same relative error for all points.
    - If c_common is a 1D array of length n: per-point relative errors (still fully correlated, rho=1).
    - If c_common is a 2D array/list with shape (k, n): k fully-correlated sources; sum their covariances.
    Returns an (n x n) covariance matrix C.
    """
    R = np.asarray(R, float).reshape(-1)
    n = R.size

    if c_common is None:
        return np.zeros((n, n), dtype=float)

    c_arr = np.asarray(c_common, dtype=float)

    # Case 1: scalar
    if c_arr.ndim == 0:
        v = R * float(c_arr)               # absolute fully-correlated error per point
        return np.outer(v, v)

    # Case 2: 1D vector of length n
    if c_arr.ndim == 1:
        if c_arr.size != n:
            raise ValueError("c_common vector must have same length as R")
        v = R * c_arr
        return np.outer(v, v)

    # Case 3: 2D array: rows are different fully-correlated sources
    if c_arr.ndim == 2:
        if c_arr.shape[1] != n:
            raise ValueError("c_common 2D must have shape (k, n)")
        C = np.zeros((n, n), dtype=float)
        for row in c_arr:
            v = R * row
            C += np.outer(v, v)
        return C

    raise ValueError("Unsupported shape for c_common")

def combine_ratio_with_correlated_norm(R, s_stat, u_unc, c_common=None, use_C_in_weights=False):
    """
    Combine multiple measurements of R with:
      - uncorrelated statistical relative errors s_stat (array-like)
      - uncorrelated systematic relative errors u_unc (array-like)
      - fully-correlated *multiplicative* relative error(s) c_common:
          * scalar: same for all points
          * 1D array (len n): per-point relative but fully correlated (rho=1)
          * 2D array (k x n): k fully-correlated sources; summed

    Returns a CombinedResult with full stat/syst breakdown.
    """
    R = np.asarray(R, dtype=float).reshape(-1)
    s = np.asarray(s_stat, dtype=float).reshape(-1)
    u = np.asarray(u_unc, dtype=float).reshape(-1)
    n = R.size
    if not (s.size == n and u.size == n):
        raise ValueError("R, s_stat, u_unc must have the same length")

    # Absolute uncorrelated pieces
    sig_stat = s * R
    sig_unc  = u * R
    S = np.diag(sig_stat**2)
    U = np.diag(sig_unc**2)

    # Fully-correlated multiplicative piece(s)
    C = _build_common_cov(R, c_common)

    # Full covariance
    # V = S + U + C
    V = S + U if not use_C_in_weights else (S + U + C)

    # BLUE weights
    one = np.ones(n)
    Vinv = np.linalg.inv(V)
    w = Vinv @ one / (one @ Vinv @ one)

    # Combined central value
    R_comb = float(w @ R)

    # Error decomposition with same weights
    var_stat = float(w @ S @ w)
    var_unc  = float(w @ U @ w)
    var_com  = float(w @ C @ w)   # if c_common is a vector: equals (sum_i w_i R_i c_i)^2

    sigma_stat = np.sqrt(var_stat)
    sigma_sys_unc = np.sqrt(var_unc)
    sigma_sys_com = np.sqrt(var_com)
    sigma_sys_tot = np.sqrt(var_unc + var_com)
    sigma_tot = np.sqrt(var_stat + var_unc + var_com)

    return CombinedResult(
        R_comb=R_comb,
        weights=w,
        sigma_stat=sigma_stat,
        sigma_sys_unc=sigma_sys_unc,
        sigma_sys_com=sigma_sys_com,
        sigma_sys_tot=sigma_sys_tot,
        sigma_tot=sigma_tot,
    )




In [3]:
def decompose_common_sources(R, weights, c_common, source_labels=None):
    """
    Decompose sigma_sys_com into contributions from each fully-correlated source.

    R            : array of central values (length n)
    weights      : BLUE weights used for the final combination (length n)
    c_common     : (k x n) array (or list of lists), each row = per-point *relative* error of source k
                   (still fully correlated, rho=1 for every source)
    source_labels: optional list of k labels (strings)

    Returns:
      sigmas_k      : array (k,) of absolute sigma from each source (std dev, not variance)
      sigma_com_tot : scalar, sqrt(sum_k sigmas_k^2), should match resB.sigma_sys_com
      fracs_k       : array (k,) of variance fractions (sigmas_k^2 / sum sigmas_k^2)
    """
    R = np.asarray(R, float).reshape(-1)
    w = np.asarray(weights, float).reshape(-1)
    C = np.asarray(c_common, float)

    if C.ndim == 1:
        C = C[None, :]  # make it (1, n)

    if C.shape[1] != R.size:
        raise ValueError("c_common must have shape (k, n) where n == len(R)")

    # For a fully-correlated multiplicative source k:
    # variance contribution = ( sum_i w_i * R_i * c_{k,i} )^2
    # -> std dev contribution = abs( sum_i w_i * R_i * c_{k,i} )
    t = w * R                      # length-n helper
    sigmas_k = np.abs(C @ t)       # length-k
    var_k = sigmas_k**2
    sigma_com_tot = float(np.sqrt(np.sum(var_k)))
    fracs_k = var_k / np.sum(var_k)

    # Pretty print
    if source_labels is None:
        source_labels = [f"source #{i+1}" for i in range(C.shape[0])]
    print("== Fully-correlated (multiplicative) source breakdown ==")
    for lbl, sig_k, frac in zip(source_labels, sigmas_k, fracs_k):
        print(f"  {lbl:>12s}: sigma = {sig_k:.12e}   (variance share {frac*100:6.2f}%)")
    print(f"  --> Quadrature sum (should match sigma_sys_com): {sigma_com_tot:.12e}")
    return sigmas_k, sigma_com_tot, fracs_k

In [10]:
# ---------------------------
# Br: D+ -> eta K+
# ---------------------------
# Two measurements 
R  = [1.1846e-04, 1.2692e-04]
s  = [5.5203e-06/R[0], 5.9715e-06/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.172**2 + 1.173**2 + 0.168**2 + 0.142**2 + 0.042**2)/100, math.sqrt(0.450**2 + 1.158**2 + 0.929**2 + 0.167**2 + 0.143**2 + 0.351**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.417/100, 0.433/100],  # source #1, same 1% on both
    [2.387/100, 2.387/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["(0.18%, 0.19%) common", "(2.39%, 2.39%) common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.5393603 0.4606397]
  R_comb  = 1.223570119026e-04
  sigma_stat   = 4.053578504290e-06
  sigma_sys_unc= 1.497011611165e-06
  sigma_sys_com= 2.966518552227e-06
  sigma_sys_tot= 3.322841567795e-06
  sigma_stat_sys_tot= 5.241447774719e-06


== Fully-correlated (multiplicative) source breakdown ==
  (0.18%, 0.19%) common: sigma = 5.195830422473e-07   (variance share   3.07%)
  (2.39%, 2.39%) common: sigma = 2.920661874116e-06   (variance share  96.93%)
  --> Quadrature sum (should match sigma_sys_com): 2.966518552227e-06

Cross-check:
  sigma_sys_com (from resB) = 2.966518552227e-06
  sigma_sys_com (decomposed sum) = 2.966518552227e-06


In [11]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")

# total_R1_sys_unc = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2 +  (c_sources[1][0]*R[0])**2 )
# total_R2_sys_unc = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2 +  (c_sources[1][1]*R[1])**2 )



R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)

R1_sys_norm_Br = c_sources[1][0]*R[0]
R2_sys_norm_Br = c_sources[1][1]*R[1]

total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2 + R1_sys_norm_Br**2 )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 + R2_sys_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {R1_sys_without_norm_Br:.12e} (sys without norm Br) pm {R1_sys_norm_Br:.12e} (sys norm Br)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {R2_sys_without_norm_Br:.12e} (sys without norm Br) pm {R2_sys_norm_Br:.12e} (sys norm Br)")

# print(f"x1 = {R[0]:.8e} pm (stat) pm (sys)")

Final results:
R = 1.223570119026e-04 pm 4.053578504290e-06 (stat) pm 3.322841567795e-06 (total sys)

R = 1.223570119026e-04 pm 4.053578504290e-06 (stat) pm 1.584616768103e-06 (sys without norm Br) pm 2.920661874116e-06 (sys norm Br)

R1 = 1.184600000000e-04 pm 5.520300000000e-06 (stat) pm 3.593609401433e-06 (total sys)
R2 = 1.269200000000e-04 pm 5.971500000000e-06 (stat) pm 3.692325525011e-06 (total sys)

R1 = 1.184600000000e-04 pm 5.520300000000e-06 (stat) pm 2.217764511713e-06 (sys without norm Br) pm 2.827640200000e-06 (sys norm Br)
R2 = 1.269200000000e-04 pm 5.971500000000e-06 (stat) pm 2.110665862373e-06 (sys without norm Br) pm 3.029580400000e-06 (sys norm Br)


In [12]:
# ---------------------------
# Two measurements 
R  = [3.1423e-02, 3.3667e-02]
s  = [1.4643e-03/3.1423e-02, 1.5840e-03/3.3667e-02]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.172**2 + 1.173**2 + 0.168**2 + 0.142**2 + 0.042**2)/100, math.sqrt(0.450**2 + 1.158**2 + 0.929**2 + 0.167**2 + 0.143**2 + 0.351**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.417/100, 0.433/100],  # source #1, same 1% on both
    # [2.387/100, 2.387/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["(0.18%, 0.19%) common", "(2.39%, 2.39%) common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.53149643 0.46850357]
  R_comb  = 3.247432201592e-02
  sigma_stat   = 1.075374948179e-03
  sigma_sys_unc= 3.963005367551e-04
  sigma_sys_com= 1.379416203686e-04
  sigma_sys_tot= 4.196212650263e-04
  sigma_stat_sys_tot= 1.154345392521e-03


== Fully-correlated (multiplicative) source breakdown ==
  (0.18%, 0.19%) common: sigma = 1.379416203686e-04   (variance share 100.00%)
  --> Quadrature sum (should match sigma_sys_com): 1.379416203686e-04

Cross-check:
  sigma_sys_com (from resB) = 1.379416203686e-04
  sigma_sys_com (decomposed sum) = 1.379416203686e-04


In [13]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
# print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")


R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)


total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2  )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

Final results:
R = 3.247432201592e-02 pm 1.075374948179e-03 (stat) pm 4.196212650263e-04 (total sys)

R1 = 3.142300000000e-02 pm 1.464300000000e-03 (stat) pm 5.882898383552e-04 (total sys)
R2 = 3.366700000000e-02 pm 1.584000000000e-03 (stat) pm 5.598785659352e-04 (total sys)


In [14]:
# ---------------------------
# Br: Ds+ -> eta K+
# ---------------------------
# Two measurements 
R  = [1.6164e-03, 1.6156e-03]
s  = [1.9025e-05/R[0], 2.1791e-05/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.213**2 + 1.136**2 + 0.180**2 + 0.150**2 + 0.083**2)/100, math.sqrt(0.450**2 + 0.982**2 + 1.329**2 + 0.179**2 + 0.152**2 + 0.061**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.406/100, 0.411/100],  # source #1, same 1% on both
    [1.601/100, 1.601/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["(0.19%, 0.19%) common", "(1.60%, 1.60%) common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")




Two fully-correlated sources summed:
  weights = [0.50352772 0.49647228]
  R_comb  = 1.616002822173e-03
  sigma_stat   = 1.445031907756e-05
  sigma_sys_unc= 2.036527258720e-05
  sigma_sys_com= 2.670103390967e-05
  sigma_sys_tot= 3.358108901445e-05
  sigma_stat_sys_tot= 3.655819006515e-05


== Fully-correlated (multiplicative) source breakdown ==
  (0.19%, 0.19%) common: sigma = 6.601076489067e-06   (variance share   6.11%)
  (1.60%, 1.60%) common: sigma = 2.587220518300e-05   (variance share  93.89%)
  --> Quadrature sum (should match sigma_sys_com): 2.670103390967e-05

Cross-check:
  sigma_sys_com (from resB) = 2.670103390967e-05
  sigma_sys_com (decomposed sum) = 2.670103390967e-05


In [15]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")

# total_R1_sys_unc = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2 +  (c_sources[1][0]*R[0])**2 )
# total_R2_sys_unc = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2 +  (c_sources[1][1]*R[1])**2 )



R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)

R1_sys_norm_Br = c_sources[1][0]*R[0]
R2_sys_norm_Br = c_sources[1][1]*R[1]

total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2 + R1_sys_norm_Br**2 )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 + R2_sys_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {R1_sys_without_norm_Br:.12e} (sys without norm Br) pm {R1_sys_norm_Br:.12e} (sys norm Br)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {R2_sys_without_norm_Br:.12e} (sys without norm Br) pm {R2_sys_norm_Br:.12e} (sys norm Br)")


Final results:
R = 1.616002822173e-03 pm 1.445031907756e-05 (stat) pm 3.358108901445e-05 (total sys)

R = 1.616002822173e-03 pm 1.445031907756e-05 (stat) pm 2.140837542565e-05 (sys without norm Br) pm 2.587220518300e-05 (sys norm Br)

R1 = 1.616400000000e-03 pm 1.902500000000e-05 (stat) pm 3.986689956159e-05 (total sys)
R2 = 1.615600000000e-03 pm 2.179100000000e-05 (stat) pm 3.865321902675e-05 (total sys)

R1 = 1.616400000000e-03 pm 1.902500000000e-05 (stat) pm 3.032605490255e-05 (sys without norm Br) pm 2.587856400000e-05 (sys norm Br)
R2 = 1.615600000000e-03 pm 2.179100000000e-05 (stat) pm 2.872340522428e-05 (sys without norm Br) pm 2.586575600000e-05 (sys norm Br)


In [16]:
# ---------------------------
# Br: Ds+ -> eta K+
# ---------------------------
# Two measurements 
R  = [9.5084e-02, 9.5034e-02]
s  = [1.1191e-03/R[0], 1.2818e-03/R[1]]   # stat (relative, uncorrelated)
u  = [math.sqrt(0.729**2 + 1.213**2 + 1.136**2 + 0.180**2 + 0.150**2 + 0.083**2)/100, math.sqrt(0.450**2 + 0.982**2 + 1.329**2 + 0.179**2 + 0.152**2 + 0.061**2)/100]    # syst (relative, uncorrelated)

# (B) multiple fully-correlated sources at once (each a per-point relative vector)
# e.g., luminosity [%], common calibration [%]
c_sources = np.array([
    [0.406/100, 0.411/100],  # source #1, same 1% on both
    # [1.601/100, 1.601/100],  # source #2, 1.5% vs 2.0%, still fully correlated
])
resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources, use_C_in_weights=True)
print("Two fully-correlated sources summed:")
print("  weights =", resB.weights)
print("  R_comb  =", f"{resB.R_comb:.12e}")
print("  sigma_stat   =", f"{resB.sigma_stat:.12e}")
print("  sigma_sys_unc=", f"{resB.sigma_sys_unc:.12e}")
print("  sigma_sys_com=", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_tot=", f"{resB.sigma_sys_tot:.12e}")
print("  sigma_stat_sys_tot=", f"{resB.sigma_tot:.12e}")


print("\n")

# Using your existing definitions:
# R, s, u defined above; resB = combine_ratio_with_correlated_norm(R, s, u, c_common=c_sources)

labels = ["(0.19%, 0.19%) common", "(1.60%, 1.60%) common"]
sig_k, sigma_check, fracs = decompose_common_sources(R, resB.weights, c_sources, source_labels=labels)

# Optional sanity check against the aggregator from resB
print("\nCross-check:")
print("  sigma_sys_com (from resB) =", f"{resB.sigma_sys_com:.12e}")
print("  sigma_sys_com (decomposed sum) =", f"{sigma_check:.12e}")

Two fully-correlated sources summed:
  weights = [0.5036485 0.4963515]
  R_comb  = 9.505918242486e-02
  sigma_stat   = 8.499778557132e-04
  sigma_sys_unc= 1.197979530195e-03
  sigma_sys_com= 3.882987940806e-04
  sigma_sys_tot= 1.259337487828e-03
  sigma_stat_sys_tot= 1.519339745894e-03


== Fully-correlated (multiplicative) source breakdown ==
  (0.19%, 0.19%) common: sigma = 3.882987940806e-04   (variance share 100.00%)
  --> Quadrature sum (should match sigma_sys_com): 3.882987940806e-04

Cross-check:
  sigma_sys_com (from resB) = 3.882987940806e-04
  sigma_sys_com (decomposed sum) = 3.882987940806e-04


In [17]:
print("Final results:")
print(f"R = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {resB.sigma_sys_tot:.12e} (total sys)")
# print(f"\nR = {resB.R_comb:.12e} pm {resB.sigma_stat:.12e} (stat) pm {math.sqrt(resB.sigma_sys_tot**2 - sig_k[1]**2):.12e} (sys without norm Br) pm {sig_k[1]:.12e} (sys norm Br)")


R1_sys_without_norm_Br = math.sqrt( (u[0]*R[0]) **2 + (c_sources[0][0]*R[0])**2)
R2_sys_without_norm_Br = math.sqrt( (u[1]*R[1]) **2 + (c_sources[0][1]*R[1])**2)


total_R1_sys_unc = math.sqrt( R1_sys_without_norm_Br**2  )
total_R2_sys_unc = math.sqrt( R2_sys_without_norm_Br**2 )

print(f"\nR1 = {R[0]:.12e} pm {s[0]*R[0]:.12e} (stat) pm {total_R1_sys_unc:.12e} (total sys)")
print(f"R2 = {R[1]:.12e} pm {s[1]*R[1]:.12e} (stat) pm {total_R2_sys_unc:.12e} (total sys)")


Final results:
R = 9.505918242486e-02 pm 8.499778557132e-04 (stat) pm 1.259337487828e-03 (total sys)

R1 = 9.508400000000e-02 pm 1.119100000000e-03 (stat) pm 1.783916483762e-03 (total sys)
R2 = 9.503400000000e-02 pm 1.281800000000e-03 (stat) pm 1.689589064177e-03 (total sys)


In [159]:
from dataclasses import dataclass
import math

@dataclass
class AcpCombineResult:
    A_comb: float
    w1: float
    w2: float
    sigma_stat: float
    sigma_syst_unc: float
    sigma_syst_com: float
    sigma_syst_tot: float
    sigma_tot: float

def combine_acp_two(a, b, c, d, x, y, z):
    """
    Combine two Acp-like measurements with absolute errors:
      result1 = a ± b_stat ± c_syst ± d_common
      result2 = x ± y_stat ± z_syst ± d_common
    d_common is a fully correlated additive systematic (same magnitude in both).
    Returns central value and split errors.
    """
    u1_sq = b*b + c*c  # uncorrelated variance of #1
    u2_sq = y*y + z*z  # uncorrelated variance of #2

    # BLUE weights using only uncorrelated parts
    w1 = u2_sq / (u1_sq + u2_sq)
    w2 = 1.0 - w1

    # Combined central value
    A_comb = w1*a + w2*x

    # Error components
    sigma_stat = math.sqrt((w1*b)**2 + (w2*y)**2)
    sigma_syst_unc = math.sqrt((w1*c)**2 + (w2*z)**2)
    sigma_syst_com = d  # fully correlated additive -> unchanged
    sigma_syst_tot = math.sqrt(sigma_syst_unc**2 + sigma_syst_com**2)
    sigma_tot = math.sqrt(sigma_stat**2 + sigma_syst_tot**2)

    return AcpCombineResult(
        A_comb=A_comb, w1=w1, w2=w2,
        sigma_stat=sigma_stat,
        sigma_syst_unc=sigma_syst_unc,
        sigma_syst_com=sigma_syst_com,
        sigma_syst_tot=sigma_syst_tot,
        sigma_tot=sigma_tot
    )

def print_acp_result(res: AcpCombineResult, label="Acp"):
    """Pretty-print the combination result with a clear breakdown."""
    # helper to format relative (%) if central value is non-zero
    def rel(x):
        return (x / res.A_comb * 100.0) if res.A_comb != 0 else float("nan")

    print(f"== {label} combination ==")
    print(f"Weights: w1 = {res.w1:.4f}, w2 = {res.w2:.4f}")
    print(f"{label} = {res.A_comb:.6e}")
    print(f"  stat         : {res.sigma_stat:.6e}  (rel {rel(res.sigma_stat):.3f}%)")
    print(f"  syst (uncorr): {res.sigma_syst_unc:.6e}  (rel {rel(res.sigma_syst_unc):.3f}%)")
    print(f"  syst (common): {res.sigma_syst_com:.6e}  (rel {rel(res.sigma_syst_com):.3f}%)")
    print(f"  syst (total) : {res.sigma_syst_tot:.6e}  (rel {rel(res.sigma_syst_tot):.3f}%)")
    print(f"  TOTAL        : {res.sigma_tot:.6e}     (rel {rel(res.sigma_tot):.3f}%)")
    print()
    # compact “paper-style” line
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
          f"± {res.sigma_syst_unc:.6e} (syst-unc) ± {res.sigma_syst_com:.6e} (syst-com))")
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
          f"± {res.sigma_syst_tot:.6e} (syst-unc-tot)")
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_tot:.6e})  [total]")




In [160]:
# Br: D+ -> eta pi+
a, b, c, d =  0.33e-2, 0.73e-2, math.sqrt(0.013**2 + 0.001**2 + 0.006**2)*1e-2, 7e-5   # result1: a ± b (± c ± d)
x, y, z     =  0.12e-2, 0.90e-2, math.sqrt(0.013**2 + 0.007**2 + 0.001**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two(a, b, c, d, x, y, z)
print_acp_result(res, label="A_CP")

== A_CP combination ==
Weights: w1 = 0.6031, w2 = 0.3969
A_CP = 2.466603e-03
  stat         : 5.669484e-03  (rel 229.850%)
  syst (uncorr): 1.046091e-04  (rel 4.241%)
  syst (common): 7.000000e-05  (rel 2.838%)
  syst (total) : 1.258692e-04  (rel 5.103%)
  TOTAL        : 5.670881e-03     (rel 229.906%)

A_CP = (2.466603e-03 ± 5.669484e-03 (stat) ± 1.046091e-04 (syst-unc) ± 7.000000e-05 (syst-com))
A_CP = (2.466603e-03 ± 5.669484e-03 (stat) ± 1.258692e-04 (syst-unc-tot)
A_CP = (2.466603e-03 ± 5.670881e-03)  [total]


In [161]:
# Br: Ds+ -> eta pi+
if __name__ == "__main__":
    a, b, c, d =  0.10e-2, 0.46e-2, math.sqrt(0.009**2 + 0.001**2 + 0.016**2)*1e-2, 7e-5   # result1: a ± b (± c ± d)
    x, y, z     =  -0.10e-2, 0.59e-2, math.sqrt(0.008**2 + 0.007**2 + 0.009**2)*1e-2         # result2: x ± y (± z ± d)

    res = combine_acp_two(a, b, c, d, x, y, z)
    print_acp_result(res, label="A_CP")

== A_CP combination ==
Weights: w1 = 0.6217, w2 = 0.3783
A_CP = 2.433920e-04
  stat         : 3.627707e-03  (rel 1490.479%)
  syst (uncorr): 1.258583e-04  (rel 51.710%)
  syst (common): 7.000000e-05  (rel 28.760%)
  syst (total) : 1.440150e-04  (rel 59.170%)
  TOTAL        : 3.630565e-03     (rel 1491.653%)

A_CP = (2.433920e-04 ± 3.627707e-03 (stat) ± 1.258583e-04 (syst-unc) ± 7.000000e-05 (syst-com))
A_CP = (2.433920e-04 ± 3.627707e-03 (stat) ± 1.440150e-04 (syst-unc-tot)
A_CP = (2.433920e-04 ± 3.630565e-03)  [total]


In [162]:
# Br: D+ -> eta K+
if __name__ == "__main__":
    a, b, c, d =  9.25e-2, 7.93e-2, math.sqrt(0.158**2 + 0.012**2 + 0.005**2)*1e-2, 7e-5   # result1: a ± b (± c ± d)
    x, y, z     =  2.46e-2, 8.64e-2, math.sqrt(0.152**2 + 0.004**2 + 0.052**2)*1e-2         # result2: x ± y (± z ± d)

    res = combine_acp_two(a, b, c, d, x, y, z)
    print_acp_result(res, label="A_CP")

== A_CP combination ==
Weights: w1 = 0.5428, w2 = 0.4572
A_CP = 6.145318e-02
  stat         : 5.842263e-02  (rel 95.069%)
  syst (uncorr): 1.131497e-03  (rel 1.841%)
  syst (common): 7.000000e-05  (rel 0.114%)
  syst (total) : 1.133661e-03  (rel 1.845%)
  TOTAL        : 5.843363e-02     (rel 95.086%)

A_CP = (6.145318e-02 ± 5.842263e-02 (stat) ± 1.131497e-03 (syst-unc) ± 7.000000e-05 (syst-com))
A_CP = (6.145318e-02 ± 5.842263e-02 (stat) ± 1.133661e-03 (syst-unc-tot)
A_CP = (6.145318e-02 ± 5.843363e-02)  [total]


In [ ]:
# Br: Ds+ -> eta K+
if __name__ == "__main__":
    a, b, c, d =  0.e-2, 0.e-2, math.sqrt()*1e-2, e-5   # result1: a ± b (± c ± d)
    x, y, z     =  0.e-2, 0.e-2, math.sqrt()*1e-2         # result2: x ± y (± z ± d)

    res = combine_acp_two(a, b, c, d, x, y, z)
    print_acp_result(res, label="A_CP")

In [27]:
from dataclasses import dataclass
import numpy as np

@dataclass
class AcpCombineResult:
    A_comb: float              # combined central value
    weights: np.ndarray        # BLUE weights (sum to 1)
    sigma_stat: float          # absolute statistical uncertainty
    sigma_syst_unc: float      # absolute uncorrelated systematic
    sigma_syst_com: float      # absolute fully-correlated additive systematic
    sigma_syst_tot: float      # sqrt(syst_unc^2 + syst_com^2)
    sigma_tot: float           # sqrt(stat^2 + syst_tot^2)

def _outer_sum(v_list):
    """Sum of outer products: sum_k v_k v_k^T."""
    if not v_list:
        return None
    C = np.zeros((v_list[0].size, v_list[0].size), dtype=float)
    for v in v_list:
        C += np.outer(v, v)
    return C

def build_cov_additive_common(d_common, n):
    """
    Build K for fully-correlated *additive* systematics with possibly different magnitudes.
    - d_common: None | scalar | 1D (n,) | 2D (k,n)
      values are absolute sigmas (same unit as the measurement).
    """
    if d_common is None:
        return np.zeros((n, n), dtype=float)

    d = np.asarray(d_common, dtype=float)
    if d.ndim == 0:            # scalar -> same absolute size for all points
        v = np.full(n, float(d), dtype=float)
        return np.outer(v, v)
    if d.ndim == 1:            # per-point absolute sizes
        if d.size != n:
            raise ValueError("d_common vector must have length n")
        return np.outer(d, d)  # fully correlated -> rank-1
    if d.ndim == 2:            # multiple fully-correlated additive sources (independent)
        if d.shape[1] != n:
            raise ValueError("d_common 2D must have shape (k, n)")
        v_list = [row.astype(float) for row in d]
        return _outer_sum(v_list)
    raise ValueError("Unsupported shape for d_common")

def combine_acp_two_matrix(a, b_stat, c_unc, d_common,
                           x, y_stat, z_unc):
    """
    Combine two A_CP-like measurements using full matrix calculus.
      m1 = a ± b_stat (stat) ± c_unc (uncorr syst) ± d1 (common additive)
      m2 = x ± y_stat (stat) ± z_unc (uncorr syst) ± d2 (common additive)
    Here d_common can be:
      - scalar d  -> d1=d2=d
      - vector [d1, d2] -> different magnitudes but fully correlated (rho=1)
      - 2D (k,2) -> multiple independent additive common sources; each row is [d1_k, d2_k]
    Returns AcpCombineResult with BLUE weights and an error breakdown via quadratic forms.
    """
    # data vector
    y = np.array([a, x], dtype=float)
    n = y.size

    # diagonal pieces
    S = np.diag([b_stat**2, y_stat**2])    # stat (absolute)
    U = np.diag([c_unc**2,  z_unc**2 ])    # uncorrelated syst (absolute)

    # fully-correlated additive piece(s)
    K = build_cov_additive_common(d_common, n)

    # full covariance and GLS weights
    V = S + U + K
    one = np.ones(n)
    Vinv = np.linalg.inv(V)
    w = Vinv @ one / (one @ Vinv @ one)

    # combined central value
    A_comb = float(w @ y)

    # error decomposition (all via matrices)
    var_stat = float(w @ S @ w)
    var_unc  = float(w @ U @ w)
    var_com  = float(w @ K @ w)

    sigma_stat    = np.sqrt(var_stat)
    sigma_syst_unc= np.sqrt(var_unc)
    sigma_syst_com= np.sqrt(var_com)
    sigma_syst_tot= np.sqrt(var_unc + var_com)
    sigma_tot     = np.sqrt(var_stat + var_unc + var_com)

    return AcpCombineResult(
        A_comb=A_comb,
        weights=w,
        sigma_stat=sigma_stat,
        sigma_syst_unc=sigma_syst_unc,
        sigma_syst_com=sigma_syst_com,
        sigma_syst_tot=sigma_syst_tot,
        sigma_tot=sigma_tot,
    )

# (Optional) Jacobian-based propagation check: M Vx M^T = V
def propagate_with_jacobian(b_stat, y_stat, c_unc, z_unc, d_common):
    """
    Build V via M Vx M^T with additive fully-correlated offsets:
      y = x + a * z,  z~N(0, tau^2=1),  a = [d1, d2]
    Returns V (should equal S+U+K).
    """
    S = np.diag([b_stat**2, y_stat**2])
    U = np.diag([c_unc**2,  z_unc**2 ])
    if d_common is None:
        a = np.zeros(2)
    else:
        d = np.asarray(d_common, float)
        if d.ndim == 0:
            a = np.array([float(d), float(d)], float)
        elif d.ndim == 1 and d.size == 2:
            a = d.astype(float)
        else:
            raise ValueError("Jacobian check supports scalar or 1D len-2 d_common only.")
    M  = np.column_stack([np.eye(2), a.reshape(2,1)])  # [ I | a ]
    Vx = np.zeros((3,3), float)
    Vx[:2,:2] = S + U          # diag for x1,x2
    Vx[2,2]   = 1.0            # Var(z)=tau^2
    V = M @ Vx @ M.T
    return V

def print_acp_result(res: AcpCombineResult, label="Acp"):
    """Pretty-print the combination result with a clear breakdown."""
    # helper to format relative (%) if central value is non-zero
    def rel(x):
        return (x / res.A_comb * 100.0) if res.A_comb != 0 else float("nan")

    print(f"== {label} combination ==")
    # print(f"Weights: w1 = {res.w1:.4f}, w2 = {res.w2:.4f}")
    print(f"Weights: {res.weights}")
    print(f"{label} = {res.A_comb:.6e}")
    print(f"  stat         : {res.sigma_stat:.6e}  (rel {rel(res.sigma_stat):.3f}%)")
    print(f"  syst (uncorr): {res.sigma_syst_unc:.6e}  (rel {rel(res.sigma_syst_unc):.3f}%)")
    print(f"  syst (common): {res.sigma_syst_com:.6e}  (rel {rel(res.sigma_syst_com):.3f}%)")
    print(f"  syst (total) : {res.sigma_syst_tot:.6e}  (rel {rel(res.sigma_syst_tot):.3f}%)")
    print(f"  TOTAL        : {res.sigma_tot:.6e}     (rel {rel(res.sigma_tot):.3f}%)")
    print()
    # compact “paper-style” line
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
          f"± {res.sigma_syst_unc:.6e} (syst-unc) ± {res.sigma_syst_com:.6e} (syst-com))")
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_stat:.6e} (stat) "
          f"± {res.sigma_syst_tot:.6e} (syst-unc-tot)")
    print(f"{label} = ({res.A_comb:.6e} ± {res.sigma_tot:.6e})  [total]")


In [28]:
# Br: D+ -> eta pi+
a, b, c, d =  0.49389e-2, 0.35953e-2, math.sqrt(0.020**2 + 0.0086**2 + 0.000)*1e-2, [0.000068,0.000068]   # result1: a ± b (± c ± d)
x, y, z     =  -0.06098e-2, 0.44942e-2, math.sqrt(0.022**2 + 0.0092**2 + 0.0102**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
4.938900e-03 ± 3.595300e-03 ± 2.280789e-04
eta->pipipi
-6.098000e-04 ± 4.494200e-03 ± 2.681268e-04


== A_CP combination ==
Weights: [0.60968431 0.39031569]
A_CP = 2.773155e-03
  stat         : 2.807476e-03  (rel 101.238%)
  syst (uncorr): 1.669306e-04  (rel 6.020%)
  syst (common): 6.800000e-05  (rel 2.452%)
  syst (total) : 1.802494e-04  (rel 6.500%)
  TOTAL        : 2.813256e-03     (rel 101.446%)

A_CP = (2.773155e-03 ± 2.807476e-03 (stat) ± 1.669306e-04 (syst-unc) ± 6.800000e-05 (syst-com))
A_CP = (2.773155e-03 ± 2.807476e-03 (stat) ± 1.802494e-04 (syst-unc-tot)
A_CP = (2.773155e-03 ± 2.813256e-03)  [total]


In [29]:
# Br: Ds+ -> eta pi+
a, b, c, d =  0.08267e-2, 0.23568e-2, math.sqrt(0.019**2 + 0.0086**2 + 0.0)*1e-2, [0.000068, 0.000068] # result1: a ± b (± c ± d)
x, y, z     =  -0.05604e-2, 0.30262e-2, math.sqrt(0.024**2 + 0.0092**2 + 0.0093**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
8.267000e-04 ± 2.356800e-03 ± 2.193627e-04
eta->pipipi
-5.604000e-04 ± 3.026200e-03 ± 2.816682e-04


== A_CP combination ==
Weights: [0.62253708 0.37746292]
A_CP = 3.031212e-04
  stat         : 1.859425e-03  (rel 613.426%)
  syst (uncorr): 1.658371e-04  (rel 54.710%)
  syst (common): 6.800000e-05  (rel 22.433%)
  syst (total) : 1.792372e-04  (rel 59.131%)
  TOTAL        : 1.868044e-03     (rel 616.270%)

A_CP = (3.031212e-04 ± 1.859425e-03 (stat) ± 1.658371e-04 (syst-unc) ± 6.800000e-05 (syst-com))
A_CP = (3.031212e-04 ± 1.859425e-03 (stat) ± 1.792372e-04 (syst-unc-tot)
A_CP = (3.031212e-04 ± 1.868044e-03)  [total]


In [30]:
# Br: D+ -> eta K+
a, b, c, d =  9.76922e-2, 4.48683e-2, math.sqrt(0.219**2 + 0.0203**2 + 0.0893**2)*1e-2, [0.000067, 0.000067] # result1: a ± b (± c ± d)
x, y, z     =  1.15778e-2, 4.50788e-2, math.sqrt(0.212**2 + 0.0125**2 + 0.1845**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
9.769220e-02 ± 4.486830e-02 ± 2.374710e-03
eta->pipipi
1.157780e-02 ± 4.507880e-02 ± 2.813990e-03


== A_CP combination ==
Weights: [0.50261322 0.49738678]
A_CP = 5.486004e-02
  stat         : 3.180085e-02  (rel 57.967%)
  syst (uncorr): 1.838842e-03  (rel 3.352%)
  syst (common): 6.700000e-05  (rel 0.122%)
  syst (total) : 1.840062e-03  (rel 3.354%)
  TOTAL        : 3.185404e-02     (rel 58.064%)

A_CP = (5.486004e-02 ± 3.180085e-02 (stat) ± 1.838842e-03 (syst-unc) ± 6.700000e-05 (syst-com))
A_CP = (5.486004e-02 ± 3.180085e-02 (stat) ± 1.840062e-03 (syst-unc-tot)
A_CP = (5.486004e-02 ± 3.185404e-02)  [total]


In [31]:
# Br: Ds+ -> eta K+
a, b, c, d =  2.58074e-2, 1.05740e-2, math.sqrt(0.048**2 + 0.0203**2 + 0.0667**2)*1e-2, [0.000067, 0.000067]   # result1: a ± b (± c ± d)
x, y, z     =  0.06735e-2, 1.27652e-2, math.sqrt(0.085**2 + 0.0125**2 + 0.0736**2)*1e-2         # result2: x ± y (± z ± d)

res = combine_acp_two_matrix(a, b, c, d, x, y, z)
print(f"eta->gg")
print(f"{a:.6e} ± {b:.6e} ± {math.sqrt(c**2 + d[0]**2):.6e}")
print(f"eta->pipipi")
print(f"{x:.6e} ± {y:.6e} ± {math.sqrt(z**2 + d[1]**2):.6e}")
print("\n")
print_acp_result(res, label="A_CP")

eta->gg
2.580740e-02 ± 1.057400e-02 ± 8.491095e-04
eta->pipipi
6.735000e-04 ± 1.276520e-02 ± 1.133274e-03


== A_CP combination ==
Weights: [0.59341108 0.40658892]
A_CP = 1.558823e-02
  stat         : 8.143113e-03  (rel 52.239%)
  syst (uncorr): 6.810861e-04  (rel 4.369%)
  syst (common): 6.700000e-05  (rel 0.430%)
  syst (total) : 6.843736e-04  (rel 4.390%)
  TOTAL        : 8.171820e-03     (rel 52.423%)

A_CP = (1.558823e-02 ± 8.143113e-03 (stat) ± 6.810861e-04 (syst-unc) ± 6.700000e-05 (syst-com))
A_CP = (1.558823e-02 ± 8.143113e-03 (stat) ± 6.843736e-04 (syst-unc-tot)
A_CP = (1.558823e-02 ± 8.171820e-03)  [total]


In [3]:
import math
run1 = 1/math.sqrt(428)
run2 = 1/math.sqrt(575.47)

In [7]:
(run1-run2)/run1

0.13759644042921565